In [1]:
import sys
import os
sys.path.append(os.path.abspath('..'))


In [2]:
import gradio as gr
from Day4LLMCalling import Llms, tools
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')


# openai = OpenAI(base_url="https://openrouter.ai/api/v1",api_key=os.getenv('OPENROUTER_API_KEY'))

In [3]:
import requests
from PIL import Image
from io import BytesIO
import base64
# Assuming 'client' is your OpenAI client initialized with OpenRouter's base_url

def artist(text):
    if 'generate' and 'image' not in text:
    # if not city:
        print('no image')
        return None
    
    print(f"Generating image")

    response = requests.post(
        "https://openrouter.ai/api/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {os.getenv('OPENROUTER_API_KEY')}",
            "Content-Type": "application/json",
        },
        json={
            "model": "openai/gpt-5-image-mini",
            "messages": [{"role": "user", "content": text}],
            "modalities": ["image", "text"],
        },
        timeout=120,
    )

    result = response.json()
    message = result['choices'][0]["message"]
    images = message.get("images", [])
    if images:
        img_url = images[0]["image_url"]["url"]       # data:image/png;base64,...
        image_base64 = img_url.split(",")[1]
        image_data = base64.b64decode(image_base64)
        print("Image generated!")
        return Image.open(BytesIO(image_data))
    else:
        print("No image returned")
        return None


In [4]:
from pydub import AudioSegment


def talker(text):
    if not text:
        return None

    print(f"Generating audio")

    response = requests.post(
        "https://openrouter.ai/api/v1/audio/speech",
        headers={
            "Authorization": f"Bearer {os.getenv('OPENROUTER_API_KEY')}",
            "Content-Type": "application/json",
        },
        json={
            "model": "openai/gpt-4o-mini-tts-2025-12-15",
            "voice": "onyx",
            "input": text,
            "response_format": "mp3",
        },
        timeout=60,
    )

    if response.status_code != 200:
        print(f"Error: {response.json()}")
        return None

    # Save raw bytes directly — no ffmpeg needed
    output_path = "output.mp3"
    with open(output_path, "wb") as f:
        f.write(response.content)

    print("Audio generated!")
    return output_path  # Gradio's Audio component accepts a file path

In [9]:
from Day4LLMCalling.messageSeries import mSeries
import openai.types.chat.chat_completion_message as msg

def wrapLlm(message):
    response,tool_arguments,image = Llms.callModel(message, source='openai',tools=tools,return_tool_arguments=True)
    history = mSeries.promptList.get(0,{}).get('gpt-5.4-nano',[])
    gradio_history = []
    for item in history:
        # print(type(item))
        if not (isinstance(item, msg.ChatCompletionMessage) or item.get('role') in ['tool','system'] ):
            gradio_history.append(item)
    # print(history)
    # print(gradio_history)
    if len(tool_arguments)==0:
        return response, gradio_history
    print(tool_arguments)
    return response, gradio_history, image

Input tokens: 624
Output tokens: 277
Total tokens: 901
Total calculated cost: 0.0471 cents
Generating audio


Exception in callback _ProactorBasePipeTransport._call_connection_lost(None)
handle: <Handle _ProactorBasePipeTransport._call_connection_lost(None)>
Traceback (most recent call last):
  File "C:\Users\PGCP-AI\AppData\Local\Programs\Python\Python312\Lib\asyncio\events.py", line 88, in _run
    self._context.run(self._callback, *self._args)
  File "C:\Users\PGCP-AI\AppData\Local\Programs\Python\Python312\Lib\asyncio\proactor_events.py", line 165, in _call_connection_lost
    self._sock.shutdown(socket.SHUT_RDWR)
ConnectionResetError: [WinError 10054] An existing connection was forcibly closed by the remote host


Audio generated!
Generating image
Image generated!
Input tokens: 1148
Output tokens: 53
Total tokens: 1201
Total calculated cost: 0.0296 cents
[{'text': 'Wide cinematic sunset scene. A boy (approx. 10–12 years old) standing on a natural riverbank next to the river, staring toward the horizon. He is in casual clothes: dark hoodie and jeans, sneakers, hair slightly tousled, gentle silhouette with one side lit by the sunset. Pose: relaxed, hands loosely at his sides, shoulders slightly forward. Environment: tall reeds and a few rocks near his feet, distant trees along the river, subtle fog near the water. Lighting: warm golden-orange sunset with a pink-purple gradient sky, soft volumetric sun rays, highly realistic reflection on the river surface. Composition: rule of thirds, camera at eye level, shallow depth of field (35mm look), crisp focus on the boy and river reflection, background gently blurred. Mood: calm and contemplative. Style: premium realistic illustration / semi-photoreal, s

In [6]:
with gr.Blocks() as ui:
    city_state = gr.State()
    audio_state = gr.State()
    with gr.Row():
        chat_history = gr.Chatbot(height=500, label='Chat History')
        image_box = gr.Image(height=500, interactive=False, show_label=False)
    with gr.Row():
        audio_box = gr.Audio(autoplay=True, type='filepath')
    with gr.Row():
        message_box = gr.Textbox(label='Chat with AI')

        message_box.submit(
            fn = wrapLlm,
            inputs=[message_box],
            outputs=[audio_state,chat_history,image_box]
            # ).then(
            # fn=artist,
            # inputs=[city_state], 
            # outputs=[image_box]
            ).then(
            fn=talker,
            inputs=[audio_state],
            outputs=[audio_box]
            )

ui.launch()
        

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [8]:
mSeries.promptList

{0: {'gpt-5.4-nano': [{'role': 'system', 'content': ''},
   {'role': 'user', 'content': 'can you generate images?'},
   {'role': 'assistant',
    'content': 'Yes—I can generate images.\n\nTell me what you want to see (subject, style, colors, mood, and any size/aspect ratio if you care). For example:  \n“Generate a cozy cabin in a snowy forest at sunset, watercolor style, warm orange lighting.”'},
   {'role': 'user',
    'content': 'generate an image of a boy standing next to a river, staring at the sunset'},
   ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_2IpJRmdY6uU1LpPurerOvNDD', function=Function(arguments='{"text":"A boy standing next to a calm river, staring at the sunset. Cinematic composition, golden hour lighting with warm orange and pink sky reflected on the water, gentle breeze, long shadows, natural landscape: trees along the riverbank, soft clo